# 04_test_fl_flw_5_clients.ipynb

Este notebook representa la primera integración de Flower dentro del flujo de aprendizaje federado. Se mantiene la partición del dataset CIFAR-10 en cinco clientes, pero se introduce una implementación basada en la interfaz `NumPyClient` de Flower para gestionar el intercambio de parámetros del modelo y simular el comportamiento de clientes federados.

Además, se implementa por primera vez el algoritmo FedAvg para agregar los modelos entrenados localmente y construir un modelo global. Durante el entrenamiento de cada cliente se registran métricas de consumo energético, emisiones y tiempo de ejecución mediante CodeCarbon. Finalmente, se evalúa el modelo global resultante tras una única ronda de entrenamiento federado y agregación, validando el funcionamiento conjunto de Flower, FedAvg y la instrumentación energética.

0. Constantes e Imports

In [1]:
DATA_DIR = "../data"

DATASET = "CIFAR10"

SCENARIO = "federated"
NUM_CLIENTS = 5


BATCH_SIZE = 64

EPOCHS = 5

LEARNING_RATE = 0.001

RANDOM_SEED = 42


In [2]:
import torch

import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import datasets
from torchvision import transforms

import numpy as np

import flwr as fl

from codecarbon import EmissionsTracker


1. Carga del Dataset y DataLoader

In [4]:

# ==================================================
# Transformaciones
# ==================================================

transform = transforms.ToTensor()

# ==================================================
# Dataset
# ==================================================

train_dataset = datasets.CIFAR10(
    root="../data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

# ==================================================
# Partición FL-5
# ==================================================

client_size = len(train_dataset) // NUM_CLIENTS

client_datasets = random_split(
    train_dataset,
    [client_size] * NUM_CLIENTS
)

# ==================================================
# DataLoaders de clientes
# ==================================================

client_loaders = [
    DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    for ds in client_datasets
]

# ==================================================
# Test loader global
# ==================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


2. Modelo

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

3. Modelo local para pruebas

In [6]:
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

4. Función de entrenamiento

In [7]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in train_loader:

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {epoch_loss:.4f}"
        )

5. Función de evaluación

In [8]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = correct / total

    return accuracy

6. Cliente Flower con set_parameters

In [9]:
from codecarbon import EmissionsTracker
import flwr as fl

class FlowerClient(fl.client.NumPyClient):

    def __init__(
        self,
        model,
        train_loader,
        test_loader
    ):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader

    def get_parameters(
        self,
        config
    ):
        return [
            val.detach().numpy()
            for _, val in self.model.state_dict().items()
        ]
    
    def set_parameters(
        self,
        parameters
    ):
        
        params_dict = zip(
            self.model.state_dict().keys(),
            parameters
        )

        state_dict = {
            k: torch.tensor(v)
            for k, v in params_dict
        }

        self.model.load_state_dict(
            state_dict,
            strict=True
        )

    def fit(
        self,
        parameters,
        config
    ):
        self.set_parameters(parameters)

        local_criterion = nn.CrossEntropyLoss()

        local_optimizer = optim.Adam(
            self.model.parameters(),
            lr=LEARNING_RATE
        )

        tracker = EmissionsTracker(
            save_to_file=False
        )

        tracker.start()

        tracker.start_task("training")

        train_model(
            self.model,
            self.train_loader,
            local_optimizer,
            local_criterion,
            EPOCHS
        )

        training_data = tracker.stop_task()

        tracker.stop()

        metrics = {
            "energy_kwh": training_data.energy_consumed,
            "emissions_kg": training_data.emissions,
            "duration_s": training_data.duration
        }

        return (
            self.get_parameters(config),
            len(self.train_loader.dataset),
            metrics
        )
    
    def evaluate(
        self,
        parameters,
        config
    ):
        return (
            0.0,
            len(self.test_loader.dataset),
            {"accuracy": 0.0}
        )

7. Función FedAvg

In [10]:
def fedavg(client_parameters):

    aggregated_parameters = []

    for layer_weights in zip(*client_parameters):

        aggregated_layer = np.mean(
            layer_weights,
            axis=0
        )

        aggregated_parameters.append(
            aggregated_layer
        )

    return aggregated_parameters

In [11]:
## Obtener pesos iniciales

global_model = SimpleCNN()

global_parameters = [
    val.detach().numpy()
    for _, val in global_model.state_dict().items()
]

In [12]:
# Crear clientes de Flower

clients = []

for i in range(NUM_CLIENTS):

    clients.append(
        FlowerClient(
            model=SimpleCNN(),
            train_loader=client_loaders[i],
            test_loader=test_loader
        )
    )

In [13]:
# Entrenamiento federado de los 5 clientes

client_parameters = []

client_metrics = []

for client in clients:

    updated_parameters, num_examples, metrics = client.fit(
        global_parameters,
        {}
    )

    client_parameters.append(
        updated_parameters
    )

    client_metrics.append(
        metrics
    )

[codecarbon WARNING @ 13:30:35] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 13:30:35] [setup] RAM Tracking...
[codecarbon INFO @ 13:30:35] [setup] CPU Tracking...
[codecarbon INFO @ 13:30:35] [setup] GPU Tracking...
[codecarbon INFO @ 13:30:35] No GPU found.
[codecarbon WARNING @ 13:30:37] We saw that you have a 12th Gen Intel(R) Core(TM) i7-1255U but we don't know it. Please contact us.
[codecarbon WARNING @ 13:30:37] We will use the default power consumption of 4 W per thread for your 12 CPU, so 48W.
[codecarbon WARNING @ 13:30:37] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 13:30:37] CPU Model on constant consumption mode: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon WARNING @ 13:30:37] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon

Epoch 1/5 - Loss: 1.8721
Epoch 2/5 - Loss: 1.4638
Epoch 3/5 - Loss: 1.3127
Epoch 4/5 - Loss: 1.2100


[codecarbon INFO @ 13:31:08] Energy consumed for RAM : 0.000085 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:31:08] Delta energy consumed for CPU with cpu_load : 0.000082 kWh, power : 9.618701155354838 W
[codecarbon INFO @ 13:31:08] Energy consumed for All CPU : 0.000082 kWh
[codecarbon INFO @ 13:31:08] 0.000168 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 13:31:08] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 13:31:08] >>> Tracker's metadata:
[codecarbon INFO @ 13:31:08]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 13:31:08]   Python version: 3.12.3
[codecarbon INFO @ 13:31:08]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 13:31:08]   Available RAM : 7.632 GB
[codecarbon INFO @ 13:31:08]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 13:31:08]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 13:31:0

Epoch 5/5 - Loss: 1.1184
Epoch 1/5 - Loss: 1.9140
Epoch 2/5 - Loss: 1.5042
Epoch 3/5 - Loss: 1.3457
Epoch 4/5 - Loss: 1.2367


[codecarbon INFO @ 13:31:38] Energy consumed for RAM : 0.000084 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:31:38] Delta energy consumed for CPU with cpu_load : 0.000080 kWh, power : 9.607960094090323 W
[codecarbon INFO @ 13:31:38] Energy consumed for All CPU : 0.000080 kWh
[codecarbon INFO @ 13:31:38] 0.000164 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 13:31:38] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 13:31:38] >>> Tracker's metadata:
[codecarbon INFO @ 13:31:38]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 13:31:38]   Python version: 3.12.3
[codecarbon INFO @ 13:31:38]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 13:31:38]   Available RAM : 7.632 GB
[codecarbon INFO @ 13:31:38]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 13:31:38]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 13:31:3

...

Epoch 5/5 - Loss: 1.1239
Epoch 1/5 - Loss: 1.8820
Epoch 2/5 - Loss: 1.5029
Epoch 3/5 - Loss: 1.3369
Epoch 4/5 - Loss: 1.2298


[codecarbon INFO @ 13:32:38] Energy consumed for RAM : 0.000083 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:32:38] Delta energy consumed for CPU with cpu_load : 0.000080 kWh, power : 9.61213622592 W
[codecarbon INFO @ 13:32:38] Energy consumed for All CPU : 0.000080 kWh
[codecarbon INFO @ 13:32:38] 0.000163 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 13:32:38] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 13:32:38] >>> Tracker's metadata:
[codecarbon INFO @ 13:32:38]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 13:32:38]   Python version: 3.12.3
[codecarbon INFO @ 13:32:38]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 13:32:38]   Available RAM : 7.632 GB
[codecarbon INFO @ 13:32:38]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 13:32:38]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 13:32:38]  

Epoch 5/5 - Loss: 1.1245
Epoch 1/5 - Loss: 1.9228
Epoch 2/5 - Loss: 1.5206
Epoch 3/5 - Loss: 1.3691
Epoch 4/5 - Loss: 1.2365


[codecarbon INFO @ 13:33:09] Energy consumed for RAM : 0.000086 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:33:09] Delta energy consumed for CPU with cpu_load : 0.000083 kWh, power : 9.6651299946 W
[codecarbon INFO @ 13:33:09] Energy consumed for All CPU : 0.000083 kWh
[codecarbon INFO @ 13:33:09] 0.000170 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 1.1404


In [14]:
# Agregación FedAvg

global_parameters = fedavg(
    client_parameters
)

8. Métricas federadas

In [15]:
energy_total = sum(
    m["energy_kwh"]
    for m in client_metrics
)

emissions_total = sum(
    m["emissions_kg"]
    for m in client_metrics
)

duration_total = sum(
    m["duration_s"]
    for m in client_metrics
)

print("Energy:", energy_total)
print("Emissions:", emissions_total)
print("Duration:", duration_total)

Energy: 0.0008266839253721543
Emissions: 0.00014388433721102347
Duration: 136.0739513249937


9. Crear modelo global

In [16]:
# Cargamos los pesos federados en el modelo global

global_model = SimpleCNN()

params_dict = zip(
    global_model.state_dict().keys(),
    global_parameters
)

state_dict = {
    k: torch.tensor(v)
    for k, v in params_dict
}

global_model.load_state_dict(
    state_dict,
    strict=True
)

<All keys matched successfully>

In [17]:
# Obtener accuracy del modelo global en el conjunto de prueba

global_accuracy = evaluate_model(
    global_model,
    test_loader
)

print(
    f"Global Accuracy: {global_accuracy:.4f}"
)

Global Accuracy: 0.5019


In [18]:
global_model = SimpleCNN()

global_parameters = [
    val.detach().numpy()
    for _, val in global_model.state_dict().items()
]

10. Prueba de validacion

In [19]:
# VALIDACIÓN DEL CLIENTE FLOWER

result = clients[0].fit(
    global_parameters,
    {}
)

print(
    "Métricas devueltas por el cliente:"
)

print(result[2])

[codecarbon WARNING @ 13:33:12] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 13:33:12] >>> Tracker's metadata:
[codecarbon INFO @ 13:33:12]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 13:33:12]   Python version: 3.12.3
[codecarbon INFO @ 13:33:12]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 13:33:12]   Available RAM : 7.632 GB
[codecarbon INFO @ 13:33:12]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 13:33:12]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 13:33:12]   GPU count: 0
[codecarbon INFO @ 13:33:12]   GPU model: 
[codecarbon INFO @ 13:33:12] Energy consumed for RAM : 0.000000 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:33:12] Delta energy consumed for CPU with cpu_load : 0.000000 kWh, power : 10.200000000000001 W
[codecarbon INFO @ 13:33:12] Energy consumed for All CPU : 0.000000 kWh
[codecarbon INFO @ 13:33:12] 0.000000 kWh of elect

Epoch 1/5 - Loss: 1.8850
Epoch 2/5 - Loss: 1.5335
Epoch 3/5 - Loss: 1.3964
Epoch 4/5 - Loss: 1.2613


[codecarbon INFO @ 13:33:41] Energy consumed for RAM : 0.000082 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 13:33:41] Delta energy consumed for CPU with cpu_load : 0.000079 kWh, power : 9.7258443672 W
[codecarbon INFO @ 13:33:41] Energy consumed for All CPU : 0.000079 kWh
[codecarbon INFO @ 13:33:41] 0.000161 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 1.1736
Métricas devueltas por el cliente:
{'energy_kwh': 0.00016107696960741437, 'emissions_kg': 2.8035446560170473e-05, 'duration_s': 26.3071687499978}
